**BUILDING A MODEL TO RATE SENTIMENTS OF TWEETS ABOUT GOOGLE AND APPLE PRODUCTS**
Student Name: Stephen Jilani Project: Phase 3 Date:14 June 2026

**1. BUSINESS UNDERSTANDING**

A Kenyan company an is in a predicament on whether to venture in Apple or Google products. The company has an objective to build a classification model that will help them identify either to venture into either Apple or Google products.

**STAKEHOLDERS**

The stakeholder of the project is the Kenyan company at large, however, the specific persons of interest include but not limited to the following:

    1. The business owner/CEO.
    2. Product managers.
    3. Sales and marketing department.
    4. Research and development department.

**BUSINESS OBJECTIVES** The primary objective is to build a classification model on twitter reviews on Google and Apple products to see if there are any patterns for which products the business can venture into.

    1. Analyse customer sentiment towards Apple and Google products using Twitter reviews
    2. Support inventory and product selection decisions with data driven insights
    3. Minimize risk of stocking products with negative customer sentiment

**BUSINESS QUESTIONS**

    1. Which brand has the highest proportion of positive reviews?
    2. How are the reviews distributed across positive, negative and neutral sentiment categories?
    3. Which customer concerns are mostly frequently associated with negative sentiment?

**PROJECT SUCCESS METRICS**

This project is going to be a success if the metrics below are met:

    1. If the reviews can be successfully classified as positive, negative or neutral
    2. If actionable insights can be generated on the brands in question
    3. If there is ability fir the business to compare the 2 brands in terms of their proportion of positive reviews.

**2. DATA UNDERSTANDING**

The data was obtained from CriwdFlower via data.world, and is composed of sentiments which were rated by human raters as positive, negative or either in over 9,000 tweets. The sentiments are in relation to Apple and Google products.

Preliminary analysis on the data for better understanding of the data.

In [1]:
#Importation of necesarry libraries
import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

In [2]:
#Loading the dataset
df = pd.read_csv('judge-1377884607_tweet_product_company.csv', encoding="latin1")
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [3]:
#Checking descriptive statistics on the data
df.info()

#Observations
#The data has 3 columns and 9,003 rows. The colums are for the tweet text, the brand directed to and the sentiment classification. The columns will need
# renamed to make their names shorter.
#All the 3 columns are formated in text

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


In [4]:
#Checking the columns on brands and sentiments to see the unique entries there. The tweent column is expected to have unique entries.However, we will
#check just to be sure.

print(df['emotion_in_tweet_is_directed_at'].unique())
print(df['emotion_in_tweet_is_directed_at'].value_counts())
print(df['is_there_an_emotion_directed_at_a_brand_or_product'].unique())
print(df['is_there_an_emotion_directed_at_a_brand_or_product'].value_counts())

#Observations
#The "brand column has quite a number of unique entries but from the results below. There is a possibility that the entries there can be transformed into
#either Apple or Google. The same can also be said about the "sentiment classification" column which has less unique items but all seem to be able to
#be classified as either Negative, Positive or Neutral via data cleaning.

['iPhone' 'iPad or iPhone App' 'iPad' 'Google' nan 'Android' 'Apple'
 'Android App' 'Other Google product or service'
 'Other Apple product or service']
emotion_in_tweet_is_directed_at
iPad                               946
Apple                              661
iPad or iPhone App                 470
Google                             430
iPhone                             297
Other Google product or service    293
Android App                         81
Android                             78
Other Apple product or service      35
Name: count, dtype: int64
['Negative emotion' 'Positive emotion'
 'No emotion toward brand or product' "I can't tell"]
is_there_an_emotion_directed_at_a_brand_or_product
No emotion toward brand or product    5389
Positive emotion                      2978
Negative emotion                       570
I can't tell                           156
Name: count, dtype: int64


In [5]:
#Checking for blank columns in the dataset
df.isna().sum()

#Observation
#The tweet text has one blank row. This may have to be dropped as the lack of text measn there is nothing to be rated.
#The brand column has over 50% of blank rows and this is significant hence may need to be filled based on the condition of the tweet
#The emotion column has no  blank entries which is a good thing.

tweet_text                                               1
emotion_in_tweet_is_directed_at                       5802
is_there_an_emotion_directed_at_a_brand_or_product       0
dtype: int64

In [6]:
#Checking if the tweet text has any duplicates as this is expected to have unique entries only.
df['tweet_text'].duplicated().sum()
#This observation is an indication of possibility that there are reviews that were submitted twice. The possibility of having 2 people submiting an indentical review
#on a product arevery minimal. This will be sorted out by dropping the duplicates.

27

In [7]:
#Running the columns of the dataset to see them before renaming
df.columns

Index(['tweet_text', 'emotion_in_tweet_is_directed_at',
       'is_there_an_emotion_directed_at_a_brand_or_product'],
      dtype='object')

In [8]:
#Renaming the columns of the dataset and running the dataset to see if the names were replaced
new_columns = ['Tweet_text','Brand', 'Classification']
df.columns = new_columns
df.head()

,Tweet_text,Brand,Classification
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [9]:
#The Tweet_text column has one blank row which will be dropped. We will also remove any white spaces or empty strings if any.
df = df.dropna(subset =['Tweet_text'])
df = df[df['Tweet_text'].str.strip() !=""]

In [10]:
#Printing info for the Tweet_text column to see if the rows have been dropped.
df['Tweet_text'].shape
#Observation
#The column initially had 9,003 rows but now 1 has been dropped.

(9092,)

In [11]:
# Dropping the duplicates in the tweet_text column. This is because they might be reviews which have been entered more than once as its almost
#impossible for2 people to give an identical review in form of a tweet.

df = df.drop_duplicates(subset = ['Tweet_text'])
df['Tweet_text'].shape

(9065,)

In [12]:
#We will now go to the classification column and ensure it's cleaned then we will go back to the tweet_text column later on.

df['Classification'].unique()

#The column has emotions like "I can't tell" and 'No emotion toward brand or product' which are the ones that can be considered as neutral.
#We will replace these strings with 'neutral"

array(['Negative emotion', 'Positive emotion',
       'No emotion toward brand or product', "I can't tell"], dtype=object)

In [15]:
df['Classification']= df['Classification'].replace({'No emotion toward brand or product':'Neutral emotion', "I can't tell":'Neutral emotion'})
df['Classification'].unique()

array(['Negative emotion', 'Positive emotion', 'Neutral emotion'],
      dtype=object)

In [16]:
df[df['Brand'].isna()]['Tweet_text'].head(20)

5     @teachntech00 New iPad Apps For #SpeechTherapy...
16    Holler Gram for iPad on the iTunes App Store -...
32    Attn: All  #SXSW frineds, @mention Register fo...
33        Anyone at  #sxsw want to sell their old iPad?
34    Anyone at  #SXSW who bought the new iPad want ...
35    At #sxsw.  Oooh. RT @mention Google to Launch ...
37    SPIN Play - a new concept in music discovery f...
39    VatorNews - Google And Apple Force Print Media...
41    HootSuite - HootSuite Mobile for #SXSW ~ Updat...
42    Hey #SXSW - How long do you think it takes us ...
43    Mashable! - The iPad 2 Takes Over SXSW [VIDEO]...
44    For I-Pad ?RT @mention New #UberSocial for #iP...
46    Hand-Held Û÷HoboÛª: Drafthouse launches Û÷H...
48    Orly....? ÛÏ@mention Google set to launch new...
50    Khoi Vinh (@mention says Conde Nast's headlong...
51    ÛÏ@mention {link} &lt;-- HELP ME FORWARD THIS...
52    ÷¼ WHAT? ÷_ {link} ã_ #edchat #musedchat #s...
53    .@mention @mention on the location-based '

In [ ]:
#